# Final Forecasting & Submission

In this notebook, we will train the finalized forecasting models, generate predictions for all 14 test forecasting windows, verify the prediction output carefully, and create the final competition submission file in the required format.

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

DATA_DIR = Path("../data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_input.csv")

for df in [train, test]:
    df["datetime"] = pd.to_datetime(
        df["datetime"],
        format="%d-%m-%Y %H:%M"
    )

sys.path.append(str(Path.cwd().parent))

from src.features import (
    build_features,
    get_model_features,
    TARGETS
)

train_fe = build_features(train)
test_fe = build_features(test)

model_features = get_model_features(train_fe)

print("Train:", train_fe.shape)
print("Test:", test_fe.shape)
print("Features:", len(model_features))

Train: (34343, 118)
Test: (2352, 114)
Features: 110


In [2]:
from catboost import CatBoostRegressor

# Target-specific configs selected from validation tuning
final_configs = {
    "nat_demand": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "load_tocumen_mwh": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "load_santiago_mwh": {
        "iterations": 800,
        "depth": 5,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "load_david_mwh": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    }
}

X_train = train_fe[model_features]
X_test = test_fe[model_features]

regular_preds = {}
peak_preds = {}

for target in TARGETS:
    print(f"\nTraining: {target}")

    y = train_fe[target]

    config = final_configs[target]

    # -------------------------
    # Regular CatBoost
    # -------------------------
    regular_model = CatBoostRegressor(
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **config
    )

    regular_model.fit(X_train, y)

    regular_preds[target] = regular_model.predict(X_test)

    # -------------------------
    # Peak-weighted CatBoost
    # -------------------------
    peak_threshold = y.quantile(0.90)

    sample_weights = np.where(
        y >= peak_threshold,
        2.0,
        1.0
    )

    peak_model = CatBoostRegressor(
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **config
    )

    peak_model.fit(
        X_train,
        y,
        sample_weight=sample_weights
    )

    peak_preds[target] = peak_model.predict(X_test)

print("\nAll final models trained.")


Training: nat_demand

Training: load_tocumen_mwh

Training: load_santiago_mwh

Training: load_david_mwh

All final models trained.


In [3]:
final_predictions = test.copy()

for target in TARGETS:
    final_predictions[target] = (
        0.5 * regular_preds[target]
        + 0.5 * peak_preds[target]
    )

    # Load cannot be negative
    final_predictions[target] = np.maximum(
        final_predictions[target],
        0
    )

print(final_predictions[TARGETS].head())

    nat_demand  load_tocumen_mwh  load_santiago_mwh  load_david_mwh
0  1151.927351        950.026424          74.494324      134.189683
1  1114.020435        916.024053          71.279120      129.017759
2  1073.832102        882.768652          69.009591      124.762477
3  1055.845767        866.741571          67.595784      121.980009
4  1049.599479        862.678343          66.939477      121.244197


In [4]:
print("Rows:", len(final_predictions))
print("Columns:", len(final_predictions.columns))

print("\nTarget columns:")
print(list(final_predictions[TARGETS].columns))

print("\nMissing predictions:")
print(final_predictions[TARGETS].isna().sum())

print("\nNegative predictions:")
print((final_predictions[TARGETS] < 0).sum())

print("\nTest windows:", final_predictions["window_id"].nunique())
print("Horizon range:",
      final_predictions["horizon_hour"].min(),
      "to",
      final_predictions["horizon_hour"].max())

Rows: 2352
Columns: 51

Target columns:
['nat_demand', 'load_tocumen_mwh', 'load_santiago_mwh', 'load_david_mwh']

Missing predictions:
nat_demand           0
load_tocumen_mwh     0
load_santiago_mwh    0
load_david_mwh       0
dtype: int64

Negative predictions:
nat_demand           0
load_tocumen_mwh     0
load_santiago_mwh    0
load_david_mwh       0
dtype: int64

Test windows: 14
Horizon range: 1 to 168


In [5]:
# ============================
# FINAL SUBMISSION AUDIT
# ============================

original_test_columns = list(test.columns)

expected_columns = original_test_columns + TARGETS

assert list(final_predictions.columns) == expected_columns
assert len(final_predictions) == len(test)

# Confirm original test data was not modified
for col in original_test_columns:
    if col in TARGETS:
        continue
    assert final_predictions[col].equals(test[col]), f"Modified column: {col}"

# Prediction checks
assert final_predictions[TARGETS].isna().sum().sum() == 0
assert (final_predictions[TARGETS] < 0).sum().sum() == 0

print("✅ Column order verified")
print("✅ Original test columns preserved")
print("✅ Predictions contain no NaNs")
print("✅ Predictions are non-negative")
print("✅ Final shape:", final_predictions.shape)

# Replace with your actual team ID
TEAM_ID = "MM2611"

output_path = Path("../outputs/predictions") / f"{TEAM_ID}_MM26ML02.csv"

final_predictions.to_csv(output_path, index=False)

print(f"\n✅ Submission saved to:\n{output_path}")

✅ Column order verified
✅ Original test columns preserved
✅ Predictions contain no NaNs
✅ Predictions are non-negative
✅ Final shape: (2352, 51)

✅ Submission saved to:
..\outputs\predictions\MM2611_MM26ML02.csv


In [6]:
check = pd.read_csv(output_path)

print("Saved shape:", check.shape)
print("File size:", round(output_path.stat().st_size / 1024, 2), "KB")
print("\nPrediction summary:")
print(check[TARGETS].describe().round(2))

Saved shape: (2352, 51)
File size: 1228.39 KB

Prediction summary:
       nat_demand  load_tocumen_mwh  load_santiago_mwh  load_david_mwh
count     2352.00           2352.00            2352.00         2352.00
mean      1223.57           1003.71              77.98          141.50
std        171.90            141.02              10.99           19.91
min        814.61            664.12              51.60           94.16
25%       1078.53            886.18              68.76          125.15
50%       1212.80            993.84              77.20          140.01
75%       1358.44           1114.86              86.65          156.70
max       1583.94           1299.41             101.71          184.54


## Final Conclusion

We trained the finalized CatBoost forecasting models using the full training dataset and generated 168 hour forecasts for all 14 test windows.

The final submission uses a 50/50 blend of the regular and peak weighted CatBoost models. The output preserves all original test columns and adds the four required load predictions.

Final validation performance was approximately 3.94% average MAPE with an average R² of about 0.892. The bonus optimization layer also produced feasible schedules across all 34 validation windows, serving 100% of forecast demand with no unserved load.

The submission file passed the final structural checks: correct row count, required prediction columns, no missing predictions, and no negative forecasts.